In [1]:
!pip install -U transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 70.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 77.8 MB/s eta 0:00:00:00:01
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.0
    Uninstalling hf-xet-1.3.0:
      Successfully uninstalled hf-xet-1.3.0
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [2]:
# Test out using retrieved RAG, model PhoGPT-4B-Chat
# From a pre retrieved file, load test set from row 2068 to final row

In [3]:
from huggingface_hub import login

login("")

In [4]:
from datasets import load_dataset
dataset = load_dataset("uitnlp/vigoemotions")

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

val.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/16531 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2067 [00:00<?, ? examples/s]

In [5]:
import re

NUM_LABELS = 28  # ViGoEmotions

def parse_labels(label_str):
    indices = list(map(int, re.findall(r"\d+", label_str)))
    multi_hot = [0.0] * NUM_LABELS
    for idx in indices:
        multi_hot[idx] = 1.0
    return multi_hot

def preprocess(example):
    example["labels"] = parse_labels(example["labels"])
    return example

dataset = dataset.map(preprocess)

Map:   0%|          | 0/16531 [00:00<?, ? examples/s]

Map:   0%|          | 0/2066 [00:00<?, ? examples/s]

Map:   0%|          | 0/2067 [00:00<?, ? examples/s]

In [6]:
id2label = {
    0: "amusement",
    1: "excitement",
    2: "joy",
    3: "love",
    4: "desire",
    5: "optimism",
    6: "caring",
    7: "pride",
    8: "admiration",
    9: "gratitude",
    10: "relief",
    11: "approval",
    12: "realization",
    13: "surprise",
    14: "curiosity",
    15: "confusion",
    16: "fear",
    17: "nervousness",
    18: "remorse",
    19: "embarrassment",
    20: "disappointment",
    21: "sadness",
    22: "grief",
    23: "disgust",
    24: "anger",
    25: "annoyance",
    26: "disapproval",
    27: "neutral"
}

In [7]:
def parse_label_to_text(multi_hot):
    labels_in_text = []
    for i in range(len(multi_hot)):
        if multi_hot[i] == 1:
            labels_in_text.append(id2label[i])

    return labels_in_text

def parse_label_to_text_on_dataset(dataset):
    dataset['labels_in_text'] = parse_label_to_text(dataset['labels'])
    return dataset

dataset = dataset.map(parse_label_to_text_on_dataset)

Map:   0%|          | 0/16531 [00:00<?, ? examples/s]

Map:   0%|          | 0/2066 [00:00<?, ? examples/s]

Map:   0%|          | 0/2067 [00:00<?, ? examples/s]

In [8]:
print(dataset['train']['labels_in_text'])

Column([['realization'], ['joy', 'love', 'admiration'], ['pride', 'admiration'], ['neutral'], ['disgust', 'anger'], ...])


In [9]:
# Tiền xử lý dữ liệu text
# 1. Tự xây dựng từ điển chuẩn hóa Teencode (Manually curated dictionary)
teencode_dict = {
    # 1. Nhóm phủ định và đồng ý
    "ko": "không", "k": "không", "khong": "không", "khg": "không", "hong": "không", "hông": "không",
    "chả": "chẳng", "uk": "ừ", "ukm": "ừ", "uh": "ừ",
    "đéo": "không", "đ": "đéo", # Nhóm từ mang sắc thái tiêu cực mạnh xuất hiện trong data

    # 2. Nhóm đại từ nhân xưng (Gây lỗi phân loại nhiều nhất theo Bảng 13)
    "t": "tao", "tui": "tôi", "mik": "mình", "m": "mày", "th": "thằng",
    "c": "chị", "e": "em", "ae": "anh em", "ce": "chị em", "mn": "mọi người",
    "vk": "vợ", "ck": "chồng", "hs": "học sinh", "bs": "bác sĩ",

    # 3. Nhóm tính từ / phó từ / cảm thán thường dùng (Bảng 10 & 13)
    "z": "vậy", "zậy": "vậy", "v": "vậy", "dzậy": "vậy",
    "vs": "với", "cx": "cũng", "r": "rồi", "rùi": "rồi", "ròi": "rồi",
    "nhìu": "nhiều", "ms": "mới", "mí": "mới", "lun": "luôn", "luân": "luôn",
    "quá": "quá", "qá": "quá", "ntn": "như thế nào", "bh": "bao giờ", "bây h": "bây giờ",
    "vãi": "rất", "vl": "rất", "vcl": "rất", "đỉnh": "tuyệt", "haha": "cười", "hihi": "cười",

    # 4. Nhóm động từ và danh từ phổ biến
    "dc": "được", "đc": "được", "dk": "được", "đx": "đã", "đung": "đúng",
    "thik": "thích", "pt": "biết", "bt": "biết", "ns": "nói", "lm": "làm",
    "nt": "nhắn tin", "đg": "đang", "rep": "trả lời", "vn": "việt nam",

    # 5. Nhóm từ để hỏi
    "j": "gì", "s": "sao", "ntn": "như thế nào", "chi": "gì"
}



def normalize_social_media_text(text):
    # a. Thu gọn ký tự chữ cái kéo dài (vd: cườiiiii -> cười, đẹpppp -> đẹp)
    # Giữ lại tối đa 1 ký tự (tiếng Việt không có từ nào có 2 chữ cái giống hệt nhau đi liền trừ một số từ mượn/đặc biệt)
    text = re.sub(r'([A-Za-zàáảãạâấầẩẫậăắằẳẵặèéẻẽẹêếềểễệìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵđÀÁẢÃẠÂẤẦẨẪẬĂẮẰẲẴẶÈÉẺẼẸÊẾỀỂỄỆÌÍỈĨỊÒÓỎÕỌÔỐỒỔỖỘƠỚỜỞỠỢÙÚỦŨỤƯỨỪỬỮỰỲÝỶỸỴĐ])\1{2,}', r'\1', text)

    # b. Thu gọn dấu câu kéo dài (vd: :))))) -> :)) theo đúng mô tả của bài báo
    text = re.sub(r'(\)|\(|!|\?|\.){3,}', r'\1\1', text)

    # c. Chuẩn hóa teencode bằng từ điển
    words = text.split()
    normalized_words = []
    for w in words:
        # Tách các dấu câu dính liền với chữ để tra từ điển cho chuẩn
        clean_word = re.sub(r'[^\w\s]', '', w).lower()
        if clean_word in teencode_dict:
            # Thay thế bằng từ chuẩn nhưng cố gắng giữ lại dấu câu (nếu có)
            normalized_words.append(w.lower().replace(clean_word, teencode_dict[clean_word]))
        else:
            normalized_words.append(w)

    # Nối lại thành câu (Lưu ý: EMOJI VẪN ĐƯỢC GIỮ NGUYÊN HOÀN TOÀN)
    return " ".join(normalized_words)



In [10]:
def apply_text_preprocess(example):
    example['text'] = normalize_social_media_text(example['text'])
    return example

In [11]:
test_cases = [
    "phim này đẹpppp vãiiii lun í =))))",
    "buồn quáaaa m ơiii",
    "đúng zậy luânnn"
]
for t in test_cases:
    print(f"Gốc:      {t}")
    print(f"Cải tiến: {normalize_social_media_text(t)}\n")

Gốc:      phim này đẹpppp vãiiii lun í =))))
Cải tiến: phim này đẹp rất luôn í =))

Gốc:      buồn quáaaa m ơiii
Cải tiến: buồn quáa mày ơi

Gốc:      đúng zậy luânnn
Cải tiến: đúng vậy luôn



In [12]:
dataset = dataset.map(apply_text_preprocess)

Map:   0%|          | 0/16531 [00:00<?, ? examples/s]

Map:   0%|          | 0/2066 [00:00<?, ? examples/s]

Map:   0%|          | 0/2067 [00:00<?, ? examples/s]

In [13]:
print(dataset['train'])

Dataset({
    features: ['id', 'text', 'labels', 'labels_in_text'],
    num_rows: 16531
})


In [54]:
import os
os.environ["HF_DATASETS_DISABLE_PROGRESS_BARS"] = "1"

# load_dataset(...) will now have no progress bar

def retrieve_from_test_dataset(query_id):
    query_filtered = dataset['test'].filter(lambda example: example['id'] == query_id)
    return query_filtered['text'][0],query_filtered['labels_in_text'][0]

def retrieve_from_train_dataset(query_id):
    query_filtered = dataset['train'].filter(lambda example: example['id'] == query_id)
    return query_filtered['text'][0],query_filtered['labels_in_text'][0]

In [47]:
def build_prompt(query, similar, opposite):
    sim_block = "\n".join([
        f"- Text: {text},  Label: {label}"
        for text, label in similar
    ])

    diff_block = "\n".join([
        f"- Text: {text},  Label: {label}"
        for text, label in opposite
    ])
    prompt = f"""Objective: Classify Vietnamese social media comments into one or more of the 27 emotion categories defined by GoEmotions, plus neutral.
- - -
Label Categories:
* amusement (Giải trí) : Cảm giác buồn cười trước nội dung hài hước, thú vị.
* excitement (Hào hứng) : Cảm giác vui mừng, phấn khích trước sự kiện, thông tin tích cực.
* joy (Niềm vui) : Cảm giác hạnh phúc, vui vẻ, thoải mái trước tình huống tích cực.
* love (Tình yêu) : Bày tỏ yêu thương, gắn bó với người, vật, hoặc ý tưởng.
* desire (Mong muốn) : Cảm giác mạnh mẽ về mong muốn có được ai đó hoặc điều gì đó.
* optimism (Lạc quan) : Hy vọng, tin tưởng vào kết quả tốt đẹp cho bản thân hoặc đối tượng khác.
* caring (Quan tâm) : Thể hiện tình cảm, sự chăm sóc, hoặc hỗ trợ người khác.
* pride (Tự hào) : Hài lòng, kiêu hãnh về bản thân hoặc người khác.
* admiration (Ngưỡng mộ) : Kính trọng, khâm phục ai đó vì phẩm chất hoặc thành tựu.
* gratitude (Biết ơn) : Cảm kích, trân trọng trước sự giúp đỡ, lời nói hoặc điều tích cực.
* relief (Nhẹ nhõm) : Thở phào, giảm căng thẳng sau khi vượt qua vấn đề cho bản thân hoặc đối tượng khác.
* approval (Chấp thuận) : Đồng tình hoặc ủng hộ một ý kiến, hành động.
* realization (Nhận ra) : Hiểu hoặc nhận thức, phát giác một điều gì đó.
* surprise (Ngạc nhiên) : Bất ngờ trước thông tin hoặc sự kiện không lường trước.
* curiosity (Tò mò) : Mong muốn tìm hiểu thêm, đặt câu hỏi, thắc mắc.
* confusion (Bối rối) : Lúng túng, không rõ ràng hoặc mâu thuẫn về thông tin.
* fear (Sợ hãi) : Lo lắng, bất an trước điều nguy hiểm hoặc đáng sợ.
* nervousness (Lo lắng) : Căng thẳng, bồn chồn trước tình huống, sự kiện.
* remorse (Hối hận) : Hối tiếc, tự trách về một quyết định hoặc hành động sai lầm.
* embarrassment (Xấu hổ) : Ngượng ngùng, lúng túng trước tình huống khó xử, không phù hợp ở cá nhân hoặc đối tượng khác.
* disappointment (Thất vọng) : Hụt hẫng vì kỳ vọng không được đáp ứng.
* sadness (Buồn bã) : Thất vọng hoặc không vui.
* grief (Đau buồn) : Đau đớn hoặc buồn sâu sắc khi mất mát hoặc thất bại lớn.
* disgust (Ghê tởm) : Khó chịu, chê bai, chỉ trích, khinh bỉ với điều gì gây phản cảm.
* anger (Tức giận) : Giận dữ, bức xúc vì bất công hoặc điều không vừa ý.
* annoyance (Khó chịu) : Phiền toái trước điều gì đó không hài lòng.
* disapproval (Phản đối) : Không đồng ý, thể hiện sự không hài lòng.
* neutral (Trung tính) : bình luận không thể hiện bất kỳ cảm xúc hay thái độ độ nào cụ thể. Nếu câu được gán neutral thì sẽ không gán các nhãn khác.
- - -
Annotation Guidelines
* Only use labels in label categories.
* Select multiple labels if possible relevant.
* No explanation
* Only return the annotated labels in this format: ["label1","label2",...]
- - -
Output example
Label: ["optimism", "confusion", "nervousness"]
- - -
With these texts and labels as similar references:
{sim_block}
Annotate the following text: {query}
"""
#And the following texts as different examples:
#{diff_block}    
    return prompt

In [16]:
start_row_index = 2068
end_row_index = 4134

In [17]:
pip install -U accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 8.0 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 3.3 MB/s eta 0:00:0000:0100:01m
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
Note: you may need to restart the kernel to use updated packages.


In [20]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

quant_config = BitsAndBytesConfig(
    load_in_8bit=True
)

model_id = "vilm/vinallama-7b-chat"
tokenizer = AutoTokenizer.from_pretrained("vilm/vinallama-7b-chat")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16 # Khuyến khích dùng fp16 để tối ưu bộ nhớ
)

#model.to('cuda')

prompt = "Viết một bài thơ ngắn về Hà Nội."
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
output = model.generate(**inputs, max_new_tokens=40)

print(tokenizer.decode(output[0], skip_special_tokens=True))

#messages = [
#    {"role": "user", "content": "Who are you?"},
#]
#inputs = tokenizer.apply_chat_template(
#	messages,
#	add_generation_prompt=True,
#	tokenize=True,
#	return_dict=True,
#	return_tensors="pt",
#).to(model.device)
#
#outputs = model.generate(**inputs, max_new_tokens=40)
#print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Both `max_new_tokens` (=40) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Viết một bài thơ ngắn về Hà Nội.

Hà Nội, một thành phố cổ
Nơi tiếng cười và âm nhạc tràn ngập không gian
Những người lính gác đứng canh
Bảo vệ thành phố của họ với niềm tự hào




In [ ]:
row = df.iloc[2066]

first_id = row.iloc[0]
list1 = ast.literal_eval(row.iloc[1]) if isinstance(row.iloc[1], str) else row.iloc[1]
list2 = ast.literal_eval(row.iloc[2]) if isinstance(row.iloc[2], str) else row.iloc[2]

print("ID:", first_id)
print("List 1:", list1)
print("List 2:", list2)

In [ ]:
all_possible_labels = [
    "amusement", "excitement", "joy", "love", "desire", "optimism", "caring",
    "pride", "admiration", "gratitude", "relief", "approval", "realization",
    "surprise", "curiosity", "confusion", "fear", "nervousness", "remorse",
    "embarrassment", "disappointment", "sadness", "grief", "disgust", "anger",
    "annoyance", "disapproval", "neutral"
]

In [44]:
def get_output(prompt):
    messages = [
    {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
    	messages,
    	add_generation_prompt=True,
    	tokenize=True,
    	return_dict=True,
    	return_tensors="pt",
    ).to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=40)
    labels = extract_labels(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))
    return labels

In [33]:
def extract_labels(response_text):
    try:
        # Tìm đoạn chứa dấu ngoặc vuông ví dụ ["sadness", "anger"]
        match = re.search(r'\[.*\]', response_text)
        if match:
            return ast.literal_eval(match.group(0))
    except:
        pass
    return []

In [57]:
from transformers import logging
logging.set_verbosity_error()
import pandas as pd
import ast
df = pd.read_csv('/kaggle/input/datasets/nbmdat/retrieval-result/retrieval_results.csv')

max_similar_examples = 5

max_different_examples = 3

true_labels = []
pred_labels = []

for i in range(start_row_index, end_row_index+1):
    
        row = df.iloc[i-2]

        first_id = row.iloc[0]
        list1 = ast.literal_eval(row.iloc[1]) if isinstance(row.iloc[1], str) else row.iloc[1]
        list2 = ast.literal_eval(row.iloc[2]) if isinstance(row.iloc[2], str) else row.iloc[2]
        
        query_text, query_lables_in_text =  retrieve_from_test_dataset(first_id)

        similar_examples = []
        different_examples = []

        sim = 0
        for similar_text_id in list1:
            similar_examples.append(retrieve_from_train_dataset(similar_text_id))
            sim = sim+1
            if sim==max_similar_examples:
                break


    
        dif = 0
        for different_text_id in list2:
            different_examples.append(retrieve_from_train_dataset(different_text_id))
            dif = dif+1
            if dif==max_different_examples:
                break

        
        prompt = build_prompt(query_text,similar_examples, different_examples)

        true_labels.append(query_lables_in_text)
        pred_labels.append(get_output(prompt))
        

Filter:   0%|          | 0/2067 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16531 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16531 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16531 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16531 [00:00<?, ? examples/s]

...

Filter:   0%|          | 0/16531 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2067 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16531 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16531 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16531 [00:00<?, ? examples/s]

In [58]:
print(len(true_labels),len(pred_labels))

2067 2067


In [59]:
label2id = {v: k for k, v in id2label.items()}

In [60]:
def multi_hot(labels, label2id, num_classes=28):
    vec = [0] * num_classes
    for label in labels:
        if label in label2id:   # optional safety check
            vec[label2id[label]] = 1
    return vec

example = ["amusement", "excitement"]
encoded = multi_hot(example, label2id, len(id2label))
print(encoded)

[1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [61]:
encoded_true = [multi_hot(s, label2id) for s in true_labels]

In [62]:
encoded_preds = [multi_hot(s, label2id) for s in pred_labels]

In [64]:
print(len(encoded_true),len(encoded_preds))

2067 2067


In [65]:
import pandas as pd

df = pd.DataFrame({
    "true_labels_text": true_labels,
    "pred_labels_text": pred_labels,
    "true_labels_multi_hot": encoded_true,
    "pred_labels_multi_hot": encoded_preds,
})

In [66]:
output_path = "/kaggle/working/multihot_output.xlsx"
df.to_excel(output_path, index=False)

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-generation", model="vinai/PhoGPT-4B-Chat", trust_remote_code=True)
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)